# Chapter 1 — Reading data · Notebook to complete

*Maths for AI, step by step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch01_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells you need to edit are marked **TO COMPLETE**. Run the cells in order, from top to bottom: as long as an answer is missing, the check cells say so without raising an error. The solution is in `ch01_corrige.ipynb`, on the same companion page.

**Convention.** As in the book, the variance, the standard deviation, the z-score and the covariance divide by $n$ (`ddof=0`), unless stated otherwise.

**Contents.** Data · The lesson calculations · Exercises 1.1 to 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# The book's colours: one colour = one role (observed, predicted, deviation, colleague's line)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D0.1, D1.1…) are those of the book and of the CSV files.

Variable and column names in the code are kept in French: `logements` / `logement` = homes / home, `surface_m2` = floor area in m², `prix_kEUR` = price in k€, `nord` / `sud` = North / South districts, `registre` = sales register, `vente` = sale, `pieces` = rooms, `dpe` = energy rating class, `delai_jours` = days to sell, `etage` = floor, `delais_agences` = agencies' selling times, `tilleuls` / `port` = the two residences (Les Tilleuls, Le Port), `loyers` = rents, `moyenne` = mean, `ecart(s)` = deviation(s).

In [ ]:
# Chapter 1 data, copied from the book's CSV files (do not modify)

# D0.1 — the six homes of the opening
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — sale prices of the T2 flats (two rooms, k€), lesson 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — register of sales in the South district, lessons 1.2 to 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — price per m² by floor (k€/m²), lesson 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — selling times of three agencies (days), exercise 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — monthly rents of the studios (€), exercise 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Tools provided: you do not need to modify them

def quartiles_livre(valeurs):
    """Q1 and Q3 by the book's rule (equation 1.5): ranks ⌈n/4⌉ and ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # n/4 rounded up to the next integer
    rang_q3 = -(-3 * n // 4)      # 3n/4 rounded up to the next integer
    return x[rang_q1 - 1], x[rang_q3 - 1]   # rank 1 = position 0 in Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """One row of dots per series, as in the figures of lesson 1.1.
    bandes: dictionary {name: (mean, standard deviation)} to draw mean ± standard deviation."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## The lesson calculations, in Python

These cells redo the calculations of lessons 1.1 to 1.5. Run them and compare with the book. Here, `np.std` and `np.var` are called with `ddof=0`, that is, with a division by $n$, as in the chapter.

In [ ]:
# Lesson 1.1 — mean, variance and standard deviation (division by n)
for nom, serie in [('North', nord), ('South', sud)]:
    print(f'{nom}: mean = {np.mean(serie):g} k€, variance = {np.var(serie, ddof=0):g} k€², '
          f'standard deviation = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Lesson 1.2 — counts per energy rating class (ordinal qualitative variable)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lesson 1.3 — mean, median and quartiles of the selling times
d = registre['delai_jours']
print('mean:', d.mean(), ' median:', d.median())
print('without S13: mean', d[registre['vente'] != 'S13'].mean(),
      ' median', d[registre['vente'] != 'S13'].median())
print("quartiles, the book's rule (equation 1.5):", quartiles_livre(d))
print('quartiles, NumPy default (interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lesson 1.4 — standardized scores (equation 1.6, standard deviation with division by n)
for nom, serie, vente in [('North', nord, 188), ('South', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom}: z of the series = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z of the sale at {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Lesson 1.5 — covariance and correlation of the five T2 flats in the South (division by n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r from np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r floor / price per m²:', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Exercise 1.1 — Reading a chart

*Data D1.E1. Concepts: mean and standard deviation (lesson 1.1).* The chart below reproduces the one in exercise 1.1 of the book.

In [ ]:
points_par_rangee({f'Agency {a}': v for a, v in delais_agences.items()},
                  'Selling time (days)')
plt.xlim(0, 60)
plt.show()

**a) and b)** Estimate by eye the mean selling time of each agency, then rank the agencies from the least spread out to the most spread out.

In [ ]:
# TO COMPLETE: your estimates by eye, before any calculation
moyenne_estimee = {'P': None, 'Q': None, 'R': None}   # in days
ordre_dispersion = None   # list, from least to most spread out, e.g. ['Q', 'P', 'R']

In [ ]:
# Check (division by n)
for a, v in delais_agences.items():
    print(f'Agency {a}: mean = {np.mean(v):g} d, standard deviation = {np.std(v, ddof=0):.2f} d')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Complete the previous cell.')
else:
    print('Spread order:', 'correct' if ordre_dispersion == ordre_reel else 'check again')

**c)** For Q and for R, count the sales closed within 2 days or less of the mean.

In [ ]:
# TO COMPLETE: your counts, read from the chart
proches_Q = None
proches_R = None

In [ ]:
# Check
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agency {a}: not yet completed.')
    else:
        print(f"Agency {a}: {'correct' if la_votre == vrai else 'check again'}")

*Your explanation for c):* …

## Exercise 1.2 — Two residences, the same mean rent

*Data D1.E2. Concepts: mean, variance and standard deviation (lesson 1.1).* Camille cannot spend more than 570 € a month. Questions a) to d) are first done on paper, with the table of exercise 1.2 in the book. This notebook lets you check your answers, then program the standard deviation (question e) and compare two conventions (question f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monthly rent of a studio (€)")
plt.axvline(570, color='black', linestyle=':', label="Camille's budget: 570 €")
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Reading the rents

How many studios can Camille rent in each residence with 570 €, then with 620 €?

In [ ]:
# TO COMPLETE: number of affordable studios, read from the list of rents
accessibles = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Check: `<=` produces true/false values, np.sum counts the "true" ones
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'not yet completed' if la_votre is None else ('correct' if la_votre == vrai else 'check again')
        print(f'Budget {budget} €, {nom}: {etat}')

*Your answer to a), in one sentence:* …

### b) The two means

Compute them on paper, then enter your results below.

In [ ]:
# TO COMPLETE: your means, computed by hand (in €)
moyenne_tilleuls_main = None
moyenne_port_main = None

In [ ]:
# Check
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom}: complete the previous cell.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom}: {'correct' if ok else 'check again'} (your value: {la_votre} €)")

### c) Variance and standard deviation by hand (division by $n = 6$)

On paper, draw up for each residence the table of deviations from the mean and of their squares (model: the table of the step-by-step calculation in lesson 1.1). Enter the four results here.

In [ ]:
# TO COMPLETE: your results, computed by hand (division by n)
variance_tilleuls_main = None   # in €²
variance_port_main = None       # in €²
ecart_type_tilleuls_main = None # in €
ecart_type_port_main = None     # in €

In [ ]:
# Check (division by n: ddof=0)
vos_valeurs = {
    'variance Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_main, np.var(port, ddof=0)),
    'standard deviation Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'standard deviation Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom}: not yet completed.')
    else:
        print(f"{nom}: {'correct' if np.isclose(la_votre, attendu) else 'check again'}")

### d) Interpreting

*Your answer:* …

### e) Programming the standard deviation

Complete the function by following equation 1.3 of the book, with a division by $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Use neither `np.std` nor `np.var`: the goal is to write each step yourself.

In [ ]:
# Exercise 1.2 e) — TO COMPLETE
def ecart_type(valeurs):
    """Standard deviation, division by n (equation 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = None   # TO COMPLETE: equation 1.1
    ecarts = None    # TO COMPLETE: one deviation per value
    variance = None  # TO COMPLETE: mean of the squares
    return None      # TO COMPLETE: back to the original unit

<details><summary><b>Hint</b> (click if you are stuck)</summary>

With NumPy, an operation written on an array applies to every value: `x - moyenne` gives the six deviations at once, and `ecarts ** 2` their six squares. `np.sum(...)` adds up, `np.sqrt(...)` takes the square root.

</details>

In [ ]:
# Check: run this cell without modifying it
if ecart_type(tilleuls) is None:
    print('The function still returns None: complete it.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'correct' if np.isclose(obtenu, attendu) else f'check again (expected: {attendu})'
        print(f'{nom}: {obtenu:.2f} € → {etat}')

### f) Going further: dividing by $n$ or by $n - 1$

Run the cell below. Find 10.95 by hand by dividing the sum of squares by $n - 1 = 5$. Which pandas option gives the chapter's convention?

*Your answer:* …

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, default (division by n − 1):', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (division by n)         :', loyers.std(ddof=0).round(2).to_dict())

## Exercise 1.3 — A surprising sale?

*Data D1.1. Concepts: standardized score (lesson 1.4, equation 1.6).* A T2 (two-room flat) has just sold for 196 k€. Means and standard deviations: those of lesson 1.1, with division by $n$.

In [ ]:
# TO COMPLETE: your calculations, done on paper
z_nord = None
z_sud = None
prix_sud_aussi_exceptionnel = None   # in k€, question c)

In [ ]:
# Check (standard deviation with division by n)
attendus = {
    'z North': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z South': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'equivalent South price': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'not yet completed' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'check again')
    print(f'{nom}: {etat}')

*Your answer to b):* …

## Exercise 1.4 — Do floor area and price go together?

*Data D0.1 (the six homes of the opening). Concepts: covariance and correlation (lesson 1.5, equations 1.7 and 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Floor area (m²)')
plt.ylabel('Sale price (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Before any calculation: will $r$ be positive or negative? Close to 0, to 0.5 or to 1?

*Your answer:* …

**b)** Complete the lines below. Use a division by $n$ everywhere.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = None     # TO COMPLETE: (x_i − x̄)(y_i − ȳ) for each home
covariance = None   # TO COMPLETE: equation 1.7, division by n
r = None            # TO COMPLETE: equation 1.8 (standard deviations with ddof=0)

<details><summary><b>Hint</b></summary>

`x.mean()` gives the mean, `x - x.mean()` the six deviations. The product `*` of two arrays multiplies element by element. `x.std(ddof=0)` gives the standard deviation with division by $n$.

</details>

In [ ]:
# Check
if r is None or covariance is None:
    print('Complete the previous cell.')
else:
    print('covariance:', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'check again (expected: 1637.5)')
    print('r:', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'check again')

**c)** A correlation close to 1, yet deviations of 5 to 11 k€ in absolute value: is this contradictory?

*Your answer:* …

In [ ]:
print('np.cov by default (division by n − 1):', np.cov(x, y)[0, 1])
print('np.cov with bias=True (division by n) :', np.cov(x, y, bias=True)[0, 1])

## Exercise 1.5 — Predict, then check

*Data D1.1. Concepts: mean and standard deviation (lesson 1.1), median and interquartile range (lesson 1.3).* A sixth T2 sells in the North for 240 k€.

**a)** Before any calculation, write down your predictions: do the mean, the median, the standard deviation and the interquartile range increase a lot, a little or not at all?

*Your predictions:* …

In [ ]:
nord_plus = nord + [240]
print('North:', sorted(nord))
print('North with the sale at 240 k€:', sorted(nord_plus))

**b)** Complete the function `resume`. It must return the four indicators, with the standard deviation divided by $n$ and the quartiles computed by the book's rule (`quartiles_livre`, provided above). The dictionary keys stay in French: `moyenne` = mean, `médiane` = median, `écart-type` = standard deviation, `écart interquartile` = interquartile range.

In [ ]:
# TO COMPLETE
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': None,              # TO COMPLETE
        'médiane': None,              # TO COMPLETE: np.median
        'écart-type': None,           # TO COMPLETE: division by n
        'écart interquartile': None,  # TO COMPLETE: from q1 and q3
    }

In [ ]:
tableau = pd.DataFrame({'North (5 sales)': resume(nord), 'North+240 k€ (6 sales)': resume(nord_plus),
                        'South (5 sales)': resume(sud)})
if tableau.isna().any().any():
    print('Complete the function resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k}: {'correct' if ok else 'check again'}")
tableau.round(2)

In [ ]:
# The two series, modelled on the "mean ± standard deviation" figure of lesson 1.1
if tableau.isna().any().any():
    print('Complete the function resume to display the chart.')
else:
    points_par_rangee({'North': nord, 'North + 240': nord_plus},
                      'Sale price of a T2 (k€)',
                      bandes={'North': (np.mean(nord), np.std(nord, ddof=0)),
                              'North + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Have prices in the North become less predictable than those in the South?

*Your answer:* …

## Exercise 1.6 — Challenge: two lines, the same mean error

*Data D0.1. Concepts: signed deviation and absolute value (opening, lesson 1.1), standard deviation (lesson 1.1).* Line drawn by eye: predicted price = 3 × floor area + 30. Lazy line: predicted price = 222 k€ for every home.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = None          # TO COMPLETE: observed − predicted, line drawn by eye
ecarts_paresseuse = None    # TO COMPLETE: observed − predicted, lazy line

In [ ]:
# The three indicators of question b) (standard deviation: division by n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Complete the previous cell.')
else:
    indicateurs = pd.DataFrame({
        nom: {'mean of signed deviations': e.mean(),
              'mean of absolute deviations': np.abs(e).mean(),
              'standard deviation of signed deviations': e.std(ddof=0)}
        for nom, e in [('line by eye', ecarts_oeil), ('lazy line', ecarts_paresseuse)]
    })
    print('deviations, line by eye:', ecarts_oeil)
    print('deviations, lazy line :', ecarts_paresseuse)
    print('check:', 'correct' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'check again')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="line by eye")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='lazy line')
    ax.set_xlabel('Floor area (m²)')
    ax.set_ylabel('Deviation observed − predicted (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Your answers to c) and d):* …

In [ ]:
# Check of d): the standard deviation of the lazy line's deviations equals that of the prices
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))